<a href="https://colab.research.google.com/github/jameshphan-png/Coding-Exercise---Prompt-Engineering/blob/dev/Prompt_Engineering_Part_1_(Prompt_Chaining).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
============================================================
  Customer Support AI — Prompt-Chained Multi-Step Flow
  Tools used: Claude AI, ChatGPT-4o, apillm7
============================================================

PROMPT ENGINEERING ITERATION LOG
----------------------------------
Prompt 1 (Initial):
  "Generate a Python code for a customer support AI aimed at a simple E-Commerce company."
  → Result: Basic chatbot with a free-text input and a single system prompt.
  → Problem: No structure, users didn't know what to say, AI responses were vague.

Prompt 2 (Guided Flow):
  "Make it simulate a customer flow with simple prompts for the user to identify and choose."
  Sub-questions refined the design:
    2a. How should customers pick their issue? → Numbered menu (1, 2, 3...)
    2b. What categories? → Orders & Shipping, Returns & Refunds, Billing & Payments,
        Technical Support, Account Help, General Inquiry
    2c. What happens after a category is chosen? → Guided form-style follow-up questions
  → Result: Structured intake form feeding a single AI call.
  → Problem: AI had no escalation logic; all issues got the same generic "we'll help" response.

Prompt 3 (Escalation + Closing):
  "Now that the user has chosen their prompts, end it with an optional response to create
   another support ticket (or not) & close the support system after."
  → Result: Added escalation rules per issue type and clean session close / restart loop.

IMPROVEMENT SUMMARY (Before → After)
--------------------------------------
BEFORE (Prompt 1-2 version):
  - Single SYSTEM_PROMPT with no step differentiation.
  - AI received raw form data with no pre-classification.
  - Every issue had the same resolution path (generic "we'll help").
  - No escalation rules; no urgency signals.

AFTER (Prompt 3 version — this file):
  Step 1 — Issue Classification
    Uses the raw form answers to classify urgency (low / medium / high) and
    determine whether the issue can be self-served or needs human escalation.
    Output feeds Step 2.

  Step 2 — Targeted Resolution
    Receives the Step 1 classification and generates a specific, empathetic resolution.
    Prompt constrains: tone (warm, professional), what to include (order number, next
    steps), what to avoid (legal promises, refund guarantees, pricing commitments).
    Output is shown to the customer.

  Step 3 — Escalation Decision
    Evaluates the conversation so far. If escalation is needed (high-urgency issues or
    unresolved follow-ups), the prompt instructs the AI to commit to a 1-business-day
    email follow-up and summarise the ticket for the internal team.
    Output is appended to the log as an internal memo.

  Step 4 — Session Close
    Asks the customer if they want to open a new ticket or end the session.
    Clean goodbye message generated by AI; full conversation saved to JSON log.
============================================================
"""

In [2]:
#PROMPTS 1-2 (Before)
import subprocess
subprocess.run(["pip", "install", "openai", "-q"], check=True)

import os
import json
from datetime import datetime
from openai import OpenAI

# ─── Free AI Client (No API Key Required) ────────────────────────────────────

client = OpenAI(
    base_url="https://api.llm7.io/v1",
    api_key="unused"  # No real key needed
)

MODEL = "gpt-4o-mini-2024-07-18"

# ─── Company Configuration ────────────────────────────────────────────────────

COMPANY_CONFIG = {
    "name": "Acme Corp",
    "industry": "E-commerce",
    "support_email": "support@acmecorp.com",
    "support_hours": "Monday-Friday, 9 AM - 6 PM EST",
    "website": "https://www.acmecorp.com",
    "return_policy": "30-day hassle-free returns",
}

# ─── Category Menus & Guided Forms ────────────────────────────────────────────

CATEGORIES = {
    "1": "Orders & Shipping",
    "2": "Returns & Refunds",
    "3": "Billing & Payments",
    "4": "Technical Support",
    "5": "Account Help",
    "6": "General Inquiry",
}

GUIDED_FORMS = {
    "Orders & Shipping": [
        ("order_number",  "What is your order number? (e.g. ORD-12345)"),
        ("issue",         "What's the issue?\n   1. I haven't received my order\n   2. My order arrived damaged\n   3. I received the wrong item\n   4. I need to change my delivery address\n   Enter 1-4: "),
        ("contact_email", "What email is on your account?"),
    ],
    "Returns & Refunds": [
        ("order_number",  "What is your order number?"),
        ("return_reason", "Why are you returning?\n   1. Item is defective\n   2. Wrong item received\n   3. Changed my mind\n   4. Item not as described\n   Enter 1-4: "),
        ("preference",    "Would you prefer a refund or exchange? (type refund or exchange): "),
        ("contact_email", "What email is on your account?"),
    ],
    "Billing & Payments": [
        ("issue",         "What's your billing issue?\n   1. I was charged incorrectly\n   2. My payment was declined\n   3. I need a copy of my invoice\n   4. I want to update my payment method\n   Enter 1-4: "),
        ("order_number",  "Related order number? (press Enter to skip): "),
        ("contact_email", "What email is on your account?"),
    ],
    "Technical Support": [
        ("platform",      "Where are you experiencing the issue?\n   1. Website\n   2. Mobile App\n   3. Account login\n   4. Other\n   Enter 1-4: "),
        ("description",   "Briefly describe the problem you're experiencing: "),
        ("contact_email", "What email can we reach you at?"),
    ],
    "Account Help": [
        ("issue",         "What do you need help with?\n   1. I can't log in\n   2. I want to update my details\n   3. I want to delete my account\n   4. I didn't receive a verification email\n   Enter 1-4: "),
        ("contact_email", "What email is on your account?"),
    ],
    "General Inquiry": [
        ("topic",         "What would you like to know about? (briefly describe): "),
        ("contact_email", "What email can we reach you at?"),
    ],
}

SUB_LABELS = {
    "Orders & Shipping": {
        "1": "hasn't received order", "2": "order arrived damaged",
        "3": "received wrong item",   "4": "needs to change delivery address",
    },
    "Returns & Refunds": {
        "1": "item is defective",  "2": "wrong item received",
        "3": "changed their mind", "4": "item not as described",
    },
    "Billing & Payments": {
        "1": "charged incorrectly", "2": "payment was declined",
        "3": "needs invoice copy",  "4": "wants to update payment method",
    },
    "Technical Support": {
        "1": "website issue", "2": "mobile app issue",
        "3": "account login", "4": "other technical issue",
    },
    "Account Help": {
        "1": "can't log in",            "2": "wants to update details",
        "3": "wants to delete account", "4": "didn't receive verification email",
    },
}

SYSTEM_PROMPT = (
    "You are a friendly and professional customer support agent for "
    + COMPANY_CONFIG["name"] + ", an " + COMPANY_CONFIG["industry"] + " company.\n\n"
    "You will receive a structured summary of a customer's issue gathered from a guided form.\n"
    "Your job is to:\n"
    "- Greet the customer warmly and acknowledge their specific issue\n"
    "- Provide a clear, empathetic, and helpful resolution or next step\n"
    "- Reference their order number or details where relevant\n"
    "- If the issue requires human escalation, mention they'll be contacted at their email within 1 business day\n"
    "- Keep your response concise (3-5 sentences max)\n\n"
    "Company details:\n"
    "- Support email: " + COMPANY_CONFIG["support_email"] + "\n"
    "- Support hours: " + COMPANY_CONFIG["support_hours"] + "\n"
    "- Return policy: " + COMPANY_CONFIG["return_policy"]
)

# ─── Conversation Logger ──────────────────────────────────────────────────────

class ConversationLogger:
    def __init__(self, log_file="support_log.json"):
        self.log_file = log_file
        self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_data = {
            "session_id": self.session_id,
            "started_at": datetime.now().isoformat(),
            "company": COMPANY_CONFIG["name"],
            "messages": [],
        }

    def log(self, role, content):
        self.log_data["messages"].append({
            "timestamp": datetime.now().isoformat(),
            "role": role,
            "content": content,
        })

    def save(self):
        self.log_data["ended_at"] = datetime.now().isoformat()
        try:
            existing = []
            if os.path.exists(self.log_file):
                with open(self.log_file, "r") as f:
                    existing = json.load(f)
            existing.append(self.log_data)
            with open(self.log_file, "w") as f:
                json.dump(existing, f, indent=2)
            print("\n Session saved to " + self.log_file + " (ID: " + self.session_id + ")")
        except Exception as e:
            print("\n Could not save log: " + str(e))

# ─── Helpers ─────────────────────────────────────────────────────────────────

def print_divider():
    print("-" * 58)

def show_main_menu():
    print_divider()
    print("  Please select a support category:\n")
    for key, label in CATEGORIES.items():
        print("    " + key + ". " + label)
    print_divider()

def collect_form(category):
    questions = GUIDED_FORMS[category]
    answers = {}
    print("\n  Let's gather some details about your " + category + " issue.\n")
    for field, prompt in questions:
        while True:
            answer = input("  " + prompt + " ").strip()
            if answer == "" and "skip" in prompt.lower():
                answers[field] = "N/A"
                break
            if answer:
                answers[field] = answer
                break
            print("  Please enter a value (or press Enter to skip if allowed).")
    return answers

def build_summary(category, answers):
    lines = ["Customer category: " + category]
    for field, value in answers.items():
        if field in ("issue", "return_reason", "platform") and value in SUB_LABELS.get(category, {}):
            value = SUB_LABELS[category][value]
        lines.append(field.replace("_", " ").capitalize() + ": " + value)
    return "\n".join(lines)

def ask_ai(messages):
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=512,
        messages=messages,
    )
    return response.choices[0].message.content

# ─── Main Support Flow ────────────────────────────────────────────────────────

def run_support_session(logger):
    show_main_menu()

    while True:
        choice = input("  Enter number (1-6): ").strip()
        if choice in CATEGORIES:
            category = CATEGORIES[choice]
            print("\n  You selected: " + category)
            break
        print("  Please enter a number between 1 and 6.")

    answers = collect_form(category)
    summary = build_summary(category, answers)
    logger.log("user_form", summary)

    print("\n  Connecting you with our support agent...\n")
    print_divider()

    history = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": summary},
    ]

    agent_reply = ask_ai(history)
    history.append({"role": "assistant", "content": agent_reply})
    logger.log("assistant", agent_reply)

    print("Support Agent:\n\n  " + agent_reply + "\n")
    print_divider()

    print("  Need anything else? Type a follow-up question or 'done' to exit.\n")

    while True:
        follow_up = input("  You: ").strip()
        if not follow_up:
            continue
        if follow_up.lower() in ("done", "quit", "exit", "no", "nope"):
            history.append({"role": "user", "content": "That's all, thank you!"})
            closing = ask_ai(history)
            print("\nAgent: " + closing + "\n")
            break
        history.append({"role": "user", "content": follow_up})
        logger.log("user", follow_up)
        reply = ask_ai(history)
        history.append({"role": "assistant", "content": reply})
        logger.log("assistant", reply)
        print("\nAgent: " + reply + "\n")

# ─── Entry Point ─────────────────────────────────────────────────────────────

def main():
    print("=" * 58)
    print("  " + COMPANY_CONFIG["name"] + " - Customer Support")
    print("  " + COMPANY_CONFIG["support_hours"])
    print("  " + COMPANY_CONFIG["support_email"])
    print("=" * 58)

    while True:
        logger = ConversationLogger()
        try:
            run_support_session(logger)
        except Exception as e:
            print("\n Something went wrong: " + str(e))
            print("  The session will now close safely.")
        finally:
            logger.save()

        again = input("\n  Start a new support session? (yes / no): ").strip().lower()
        if again not in ("yes", "y"):
            print("\n  Thank you for contacting " + COMPANY_CONFIG["name"] + " support. Have a great day!\n")
            break

main()

  Acme Corp - Customer Support
  Monday-Friday, 9 AM - 6 PM EST
  support@acmecorp.com
----------------------------------------------------------
  Please select a support category:

    1. Orders & Shipping
    2. Returns & Refunds
    3. Billing & Payments
    4. Technical Support
    5. Account Help
    6. General Inquiry
----------------------------------------------------------
  Enter number (1-6): 5

  You selected: Account Help

  Let's gather some details about your Account Help issue.

  What do you need help with?
   1. I can't log in
   2. I want to update my details
   3. I want to delete my account
   4. I didn't receive a verification email
   Enter 1-4:  1
  What email is on your account? james.h.phan@sjsu.edu

  Connecting you with our support agent...

----------------------------------------------------------
Support Agent:

  Hello James, I'm really sorry to hear you're having trouble logging into your Acme Corp account. Let's get you back in! Please try resetting y

In [1]:
#PROMPT 3 (AFTER, MORE OPTIMAL)
import subprocess
subprocess.run(["pip", "install", "openai", "-q"], check=True)

import os
import json
from datetime import datetime
from openai import OpenAI

# ─── Free AI Client ──────────────────────────────────────────────────────────

client = OpenAI(
    base_url="https://api.llm7.io/v1",
    api_key="unused"
)
MODEL = "gpt-4o-mini-2024-07-18"

# ─── Company Configuration ───────────────────────────────────────────────────

COMPANY_CONFIG = {
    "name": "Acme Corp",
    "industry": "E-commerce",
    "support_email": "support@acmecorp.com",
    "support_hours": "Monday–Friday, 9 AM – 6 PM EST",
    "website": "https://www.acmecorp.com",
    "return_policy": "30-day hassle-free returns",
}

# ─── Category Menus & Guided Forms ───────────────────────────────────────────

CATEGORIES = {
    "1": "Orders & Shipping",
    "2": "Returns & Refunds",
    "3": "Billing & Payments",
    "4": "Technical Support",
    "5": "Account Help",
    "6": "General Inquiry",
}

GUIDED_FORMS = {
    "Orders & Shipping": [
        ("order_number", "What is your order number? (e.g. ORD-12345)"),
        ("issue",        "What's the issue?\n   1. I haven't received my order\n"
                         "   2. My order arrived damaged\n   3. I received the wrong item\n"
                         "   4. I need to change my delivery address\n   Enter 1–4: "),
        ("contact_email","What email is on your account?"),
    ],
    "Returns & Refunds": [
        ("order_number",  "What is your order number?"),
        ("return_reason", "Why are you returning?\n   1. Item is defective\n"
                          "   2. Wrong item received\n   3. Changed my mind\n"
                          "   4. Item not as described\n   Enter 1–4: "),
        ("preference",    "Would you prefer a refund or exchange? (type refund or exchange): "),
        ("contact_email", "What email is on your account?"),
    ],
    "Billing & Payments": [
        ("issue",        "What's your billing issue?\n   1. I was charged incorrectly\n"
                         "   2. My payment was declined\n   3. I need a copy of my invoice\n"
                         "   4. I want to update my payment method\n   Enter 1–4: "),
        ("order_number", "Related order number? (press Enter to skip): "),
        ("contact_email","What email is on your account?"),
    ],
    "Technical Support": [
        ("platform",     "Where are you experiencing the issue?\n   1. Website\n"
                         "   2. Mobile App\n   3. Account login\n   4. Other\n   Enter 1–4: "),
        ("description",  "Briefly describe the problem you're experiencing: "),
        ("contact_email","What email can we reach you at?"),
    ],
    "Account Help": [
        ("issue",        "What do you need help with?\n   1. I can't log in\n"
                         "   2. I want to update my details\n   3. I want to delete my account\n"
                         "   4. I didn't receive a verification email\n   Enter 1–4: "),
        ("contact_email","What email is on your account?"),
    ],
    "General Inquiry": [
        ("topic",        "What would you like to know about? (briefly describe): "),
        ("contact_email","What email can we reach you at?"),
    ],
}

SUB_LABELS = {
    "Orders & Shipping": {
        "1": "hasn't received order", "2": "order arrived damaged",
        "3": "received wrong item",   "4": "needs to change delivery address",
    },
    "Returns & Refunds": {
        "1": "item is defective",  "2": "wrong item received",
        "3": "changed their mind", "4": "item not as described",
    },
    "Billing & Payments": {
        "1": "charged incorrectly", "2": "payment was declined",
        "3": "needs invoice copy",  "4": "wants to update payment method",
    },
    "Technical Support": {
        "1": "website issue", "2": "mobile app issue",
        "3": "account login", "4": "other technical issue",
    },
    "Account Help": {
        "1": "can't log in",            "2": "wants to update details",
        "3": "wants to delete account", "4": "didn't receive verification email",
    },
}

# Issues that always require human escalation (ARE PRIORITIZED)
HIGH_ESCALATION_ISSUES = {
    "Orders & Shipping":   ["received wrong item", "order arrived damaged"],
    "Returns & Refunds":   ["item is defective", "wrong item received"],
    "Billing & Payments":  ["charged incorrectly"],
    "Account Help":        ["wants to delete account"],
}

# ─── Step-by-Step Prompts ────────────────────────────────────────────────────
#Analyzes the multi-step flow and rulesets based on prior outputs as references, to provide the optimal experience

# STEP 1 — Issue Classification (Severity)
#   What it does: Takes the raw form summary and classifies urgency + escalation need.
#   How it uses prior output: Reads the structured form answers collected from the user.
#   Constraints: Output must be ONLY valid JSON (no prose), keys: urgency, escalate, reason.
#
STEP1_CLASSIFY_PROMPT = """
You are an internal triage engine for {company} customer support.
Analyse the structured customer issue below and respond with ONLY valid JSON — no extra text.

JSON schema:
{{
  "urgency": "low" | "medium" | "high",
  "escalate": true | false,
  "reason": "<one sentence explaining your decision>"
}}

Urgency rules:
- high: financial dispute, missing order >5 days, damaged/wrong item, account deletion.
- medium: shipping delay, billing question, technical issue blocking use.
- low: general inquiry, preference change, invoice copy.

Escalate = true when urgency is high OR the issue cannot be resolved without account access.

Customer issue:
{summary is provided here}
""".strip()

#
# STEP 2 — Targeted Resolution (Importance)
#   What it does: Uses the classification (urgency + escalate flag) to craft a tailored reply.
#   How it uses prior output: Receives both the original summary AND the Step 1 JSON result.
#   Constraints: Warm but professional tone. 3–5 sentences max. Reference order number.
#   Avoid: legal promises, guaranteed refund timelines, pricing commitments.
#
STEP2_RESOLVE_PROMPT = """
You are a warm, professional customer support agent for a company

Company details:
- Support email: {support_email}
- Support hours: {support_hours}
- Return policy: {return_policy}

You have already classified this issue internally:
- Urgency: {urgency}
- Needs human escalation: {escalate}
- Reason: {reason}

Now write a customer-facing reply (3–5 sentences):
1. Greet the customer and acknowledge their specific issue by name.
2. Provide a clear, empathetic next step tailored to the urgency level.
3. If escalate=true, tell them a support specialist will contact them at their email within 1 business day.
4. If escalate=false, give a direct self-service resolution or instruction.

Do NOT: make legal promises, guarantee specific refund timelines, or mention internal urgency scores.
Do NOT: use filler phrases like "I understand how frustrating..." more than once.

Customer issue:
{summary}
""".strip()

#
# STEP 3 — Escalation Memo (internal, not shown to customer)
#   What it does: If escalation was flagged, generate a brief internal ticket memo.
#   How it uses prior output: Summarises classification + conversation for the support team.
#   Constraints: Factual, bullet-point style. Include contact email and urgency.
#
STEP3_ESCALATION_MEMO_PROMPT = """
You are writing a brief internal escalation memo for the company support team.
Be factual and concise. Use bullet points. NO add greetings or sign-offs.

Include:
- Customer email
- Category and specific issue
- Urgency level
- Summary of the conversation (what was asked / what the bot already told them)
- Recommended action for the human agent

Conversation so far:
{conversation}
""".strip()

#
# STEP 4 — Session Close
#   What it does: Generates a warm closing message after the customer says they're done.
#   How it uses prior output: References the category and resolution from the session.
#   Constraints: 1–2 sentences only, positive tone, no upselling.
#
STEP4_CLOSE_PROMPT = """
You are wrapping up a customer support session for company.
Write a warm, 1–2 sentence closing message. Reference their support category if natural.
Keep it friendly and brief.

Support category handled: {category}
""".strip()

# ─── Conversation Logger ─────────────────────────────────────────────────────
#Conversation is logged in a JSON file thereafter.

class ConversationLogger:
    def __init__(self, log_file="support_log.json"):
        self.log_file = log_file
        self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_data = {
            "session_id": self.session_id,
            "started_at": datetime.now().isoformat(),
            "company": COMPANY_CONFIG["name"],
            "messages": [],
        }

    def log(self, role, content):
        self.log_data["messages"].append({
            "timestamp": datetime.now().isoformat(),
            "role": role,
            "content": content,
        })

    def save(self):
        self.log_data["ended_at"] = datetime.now().isoformat()
        try:
            existing = []
            if os.path.exists(self.log_file):
                with open(self.log_file, "r") as f:
                    existing = json.load(f)
            existing.append(self.log_data)
            with open(self.log_file, "w") as f:
                json.dump(existing, f, indent=2)
            print("\n  Session saved to " + self.log_file + " (ID: " + self.session_id + ")")
        except Exception as e:
            print("\n  Could not save log: " + str(e))

# ─── Helpers ─────────────────────────────────────────────────────────────────
#These augment the conversation for the customer support system, this back and forth between a relay starting from the main menu.

def print_divider(char="-", width=60):
    print(char * width)

def ask_ai(messages):
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=512,
        messages=messages,
    )
    return response.choices[0].message.content.strip()

def show_main_menu():
    print_divider()
    print("  Please select a support category:\n")
    for key, label in CATEGORIES.items():
        print(f"    {key}. {label}")
    print_divider()

def collect_form(category):
    questions = GUIDED_FORMS[category]
    answers = {}
    print(f"\n  Let's gather some details about your {category} issue.\n")
    for field, prompt in questions:
        while True:
            answer = input("  " + prompt + " ").strip()
            if answer == "" and "skip" in prompt.lower():
                answers[field] = "N/A"
                break
            if answer:
                answers[field] = answer
                break
            print("  Please enter a value (or press Enter to skip if allowed).")
    return answers

def build_summary(category, answers):
    lines = [f"Customer category: {category}"]
    for field, value in answers.items():
        if field in ("issue", "return_reason", "platform") and value in SUB_LABELS.get(category, {}):
            value = SUB_LABELS[category][value]
        lines.append(field.replace("_", " ").capitalize() + ": " + value)
    return "\n".join(lines)

def should_escalate(category, answers):
    """Quick rule-based pre-check (supplements AI classification)."""
    issue_val = answers.get("issue") or answers.get("return_reason") or ""
    resolved = SUB_LABELS.get(category, {}).get(issue_val, issue_val)
    for esc_issue in HIGH_ESCALATION_ISSUES.get(category, []):
        if esc_issue in resolved:
            return True
    return False

# ─── Multi-Step Support Flow ─────────────────────────────────────────────────

def run_support_session(logger):
    # ── Category Selection ───────────────────────────────────────────────────
    show_main_menu()
    while True:
        choice = input("  Enter number (1–6): ").strip()
        if choice in CATEGORIES:
            category = CATEGORIES[choice]
            print(f"\n  You selected: {category}")
            break
        print("  Please enter a number between 1 and 6.")

    # ── Guided Form Collection ───────────────────────────────────────────────
    answers = collect_form(category)
    summary = build_summary(category, answers)
    logger.log("user_form", summary)

    print("\n  Analysing your issue...\n")
    print_divider()

    # ── STEP 1: Issue Classification ─────────────────────────────────────────
    # Takes the raw form summary → produces urgency + escalation JSON.
    step1_prompt = STEP1_CLASSIFY_PROMPT.format(
        company=COMPANY_CONFIG["name"],
        summary=summary,
    )
    step1_result_raw = ask_ai([
        {"role": "system", "content": "You are an internal triage engine. Output only valid JSON."},
        {"role": "user",   "content": step1_prompt},
    ])

    # Parse JSON; fall back gracefully if the model adds prose around it
    try:
        # Strip any accidental markdown fences
        clean = step1_result_raw.strip().strip("```json").strip("```").strip()
        classification = json.loads(clean)
    except json.JSONDecodeError:
        classification = {"urgency": "medium", "escalate": should_escalate(category, answers), "reason": "Could not parse AI classification; using rule-based fallback."}

    logger.log("step1_classification", classification)

    # ── STEP 2: Targeted Resolution ──────────────────────────────────────────
    # Uses Step 1 output (urgency, escalate, reason) + original summary → customer reply.
    step2_prompt = STEP2_RESOLVE_PROMPT.format(
        company=COMPANY_CONFIG["name"],
        industry=COMPANY_CONFIG["industry"],
        support_email=COMPANY_CONFIG["support_email"],
        support_hours=COMPANY_CONFIG["support_hours"],
        return_policy=COMPANY_CONFIG["return_policy"],
        urgency=classification.get("urgency", "medium"),
        escalate=classification.get("escalate", False),
        reason=classification.get("reason", ""),
        summary=summary,
    )

    history = [
        {"role": "system", "content": step2_prompt},
        {"role": "user",   "content": summary},
    ]
    agent_reply = ask_ai(history)
    history.append({"role": "assistant", "content": agent_reply})
    logger.log("step2_resolution", agent_reply)

    print("Support Agent:\n")
    print("  " + agent_reply + "\n")
    print_divider()

    # ── Follow-up Conversation Loop ───────────────────────────────────────────
    print("  Need anything else? Type a follow-up question or 'done' to exit.\n")
    while True:
        follow_up = input("  You: ").strip()
        if not follow_up:
            continue
        if follow_up.lower() in ("done", "quit", "exit", "no", "nope"):
            break
        history.append({"role": "user", "content": follow_up})
        logger.log("user_followup", follow_up)
        reply = ask_ai(history)
        history.append({"role": "assistant", "content": reply})
        logger.log("assistant_followup", reply)
        print(f"\n  Agent: {reply}\n")

    # ── STEP 3: Escalation Memo (internal) ───────────────────────────────────
    # Only generated when escalation was flagged. Uses the full conversation.
    if classification.get("escalate", False):
        conversation_text = "\n".join(
            f"{m['role'].upper()}: {m['content']}"
            for m in logger.log_data["messages"]
        )
        step3_prompt = STEP3_ESCALATION_MEMO_PROMPT.format(
            company=COMPANY_CONFIG["name"],
            conversation=conversation_text,
        )
        escalation_memo = ask_ai([
            {"role": "system", "content": "You write concise internal support memos."},
            {"role": "user",   "content": step3_prompt},
        ])
        logger.log("step3_escalation_memo", escalation_memo)
        print("\n  [INTERNAL] Escalation memo generated and saved to session log.")

    # ── STEP 4: Session Close ─────────────────────────────────────────────────
    # Generates a warm closing message and saves the session log.
    step4_prompt = STEP4_CLOSE_PROMPT.format(
        company=COMPANY_CONFIG["name"],
        category=category,
    )
    closing_msg = ask_ai([
        {"role": "system", "content": "You are closing a support session warmly and briefly."},
        {"role": "user",   "content": step4_prompt},
    ])
    logger.log("step4_close", closing_msg)
    print(f"\n  Agent: {closing_msg}\n")
    print_divider()

# ─── Entry Point ─────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print(f"  {COMPANY_CONFIG['name']} — Customer Support")
    print(f"  {COMPANY_CONFIG['support_hours']}")
    print(f"  {COMPANY_CONFIG['support_email']}")
    print("=" * 60)

    while True:
        logger = ConversationLogger()
        try:
            run_support_session(logger)
        except KeyboardInterrupt:
            print("\n\n  Session interrupted.")
        except Exception as e:
            print(f"\n  Something went wrong: {e}")
            print("  The session will now close safely.")
        finally:
            logger.save()

        # ── New Session Prompt (Part of Step 4 close) ────────────────────────
        again = input("\n  Would you like to open a new support ticket? (yes / no): ").strip().lower()
        if again not in ("yes", "y"):
            print(f"\n  Thank you for contacting {COMPANY_CONFIG['name']} support. Have a great day!\n")
            break

main()

  Acme Corp — Customer Support
  Monday–Friday, 9 AM – 6 PM EST
  support@acmecorp.com
------------------------------------------------------------
  Please select a support category:

    1. Orders & Shipping
    2. Returns & Refunds
    3. Billing & Payments
    4. Technical Support
    5. Account Help
    6. General Inquiry
------------------------------------------------------------
  Enter number (1–6): 5

  You selected: Account Help

  Let's gather some details about your Account Help issue.

  What do you need help with?
   1. I can't log in
   2. I want to update my details
   3. I want to delete my account
   4. I didn't receive a verification email
   Enter 1–4:  1
  What email is on your account? james.h.phan@sjsu.edu

  Analysing your issue...

------------------------------------------------------------
Support Agent:

  Subject: Re: Login Issue Assistance

Hi James,

I understand you're having trouble logging into your Acme Corp account. To help resolve this, I'll escala